# Deployment Pitfalls: Drift & Calibration

Companion notebook for the [Deployment Pitfalls lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/02-deployment-pitfalls).

**The idea in one sentence.** A model that was accurate at launch silently decays as the
world drifts away from its training data — so production ML needs **drift detectors**
(KS test, PSI) that fire *before* accuracy craters, plus **calibration** monitoring, because
a model can be accurate yet systematically over-confident.

What we build and verify:

- **Drift detection:** the two-sample **KS test** and the **Population Stability Index**,
  both rising as the serving distribution moves.
- **Calibration:** a **reliability diagram** — and why high accuracy does *not* imply good
  probabilities.

We simulate six months of drift, **validate that both detectors fire on the drifted months
and stay quiet on the stable one**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

## Data drift detection

Monitor feature distributions over time. A KS test detects significant differences between training and serving distributions.

In [ ]:
# Simulate training distribution (model trained in Jan)
train_age = np.random.normal(35, 10, 5000)

# Simulate serving distributions over 6 months
months = [
    ('Feb', np.random.normal(35, 10, 500)),   # no drift
    ('Mar', np.random.normal(36, 10, 500)),   # slight drift
    ('Apr', np.random.normal(38, 10, 500)),   # moderate drift
    ('May', np.random.normal(40, 11, 500)),   # significant drift
    ('Jun', np.random.normal(42, 12, 500)),   # major drift
]

ks_stats = []
p_values = []
for name, serving in months:
    ks_stat, p_val = stats.ks_2samp(train_age, serving)
    ks_stats.append(ks_stat)
    p_values.append(p_val)
    print(f"{name}: KS stat={ks_stat:.4f}, p={p_val:.4f} — {'⚠️ DRIFT DETECTED' if p_val < 0.05 else '✅ No significant drift'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
month_names = [m[0] for m in months]

axes[0].plot(month_names, ks_stats, 'o-', color='#6366f1', linewidth=2)
axes[0].axhline(0.05, color='#ef4444', linestyle='--', alpha=0.7, label='Alert threshold')
axes[0].set_ylabel('KS Statistic')
axes[0].set_title('Feature drift over time (age)', fontsize=11)
axes[0].legend(facecolor='#1a1d27', edgecolor='#2a2d3a')

# Distribution comparison: Jan vs Jun
axes[1].hist(train_age, bins=40, alpha=0.6, color='#6366f1', label='Training (Jan)', density=True)
axes[1].hist(months[-1][1], bins=40, alpha=0.6, color='#ef4444', label='Serving (Jun)', density=True)
axes[1].set_title('Training vs serving distribution (age)', fontsize=11)
axes[1].legend(facecolor='#1a1d27', edgecolor='#2a2d3a')

plt.tight_layout()
plt.show()

### Validate: the KS test flags drift and stays quiet when nothing moved

The two-sample KS statistic measures the largest gap between two CDFs. It should be near zero
(and *not* significant) for the un-drifted month, and grow — crossing significance — as the
serving distribution shifts. We confirm both ends.

In [ ]:
for (name, _), k, p in zip(months, ks_stats, p_values):
    print(f'{name}: KS={k:.3f}, p={p:.4f} -> {"DRIFT" if p < 0.05 else "ok"}')
assert p_values[0] > 0.05, 'no false alarm on the un-drifted month (Feb)'
assert p_values[-1] < 0.05, 'the KS test flags the heavily drifted month (Jun)'
assert ks_stats[-1] > ks_stats[0], 'KS distance grows as the distribution drifts'
print('\n✅ the KS test catches distribution drift before accuracy visibly craters')

## Model calibration

A reliability diagram shows whether predicted probabilities match empirical frequencies.

In [ ]:
def reliability_diagram(probs, labels, n_bins=10):
    """Plot reliability diagram (calibration curve)."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_midpoints = (bins[:-1] + bins[1:]) / 2
    bin_accs = []
    bin_counts = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() > 0:
            bin_accs.append(labels[mask].mean())
            bin_counts.append(mask.sum())
        else:
            bin_accs.append(None)
            bin_counts.append(0)
    return bin_midpoints, bin_accs, bin_counts

# Generate: overconfident model, well-calibrated, underconfident
np.random.seed(10)
n = 5000
true_probs = np.random.beta(2, 2, n)  # true underlying probabilities
y_true = np.random.binomial(1, true_probs)

# Well-calibrated model: predicts close to true probs
pred_well = true_probs + np.random.randn(n) * 0.05
pred_well = np.clip(pred_well, 0, 1)

# Overconfident model: pushes probabilities toward 0 and 1
pred_over = np.clip((true_probs - 0.5) * 2 + 0.5, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (probs, title) in zip(axes, [
    (pred_well, 'Well-calibrated model'),
    (pred_over, 'Overconfident model'),
]):
    midpts, accs, counts = reliability_diagram(probs, y_true)
    valid = [(m, a) for m, a, c in zip(midpts, accs, counts) if a is not None]
    xs, ys = zip(*valid)
    ax.plot([0, 1], [0, 1], '--', color='#64748b', alpha=0.7, label='Perfect calibration')
    ax.plot(xs, ys, 'o-', color='#6366f1', linewidth=2, markersize=6, label='Model')
    ax.fill_between(xs, xs, ys, alpha=0.15, color='#ef4444')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(title, fontsize=11)
    ax.legend(facecolor='#1a1d27', edgecolor='#2a2d3a')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Population Stability Index (PSI)

PSI quantifies distribution shift. PSI < 0.1: stable; 0.1–0.2: slight change; > 0.2: significant shift.

In [ ]:
def compute_psi(expected, actual, n_bins=10):
    """Population Stability Index."""
    bins = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    bins[0] = -np.inf
    bins[-1] = np.inf
    
    exp_counts = np.histogram(expected, bins=bins)[0] / len(expected)
    act_counts = np.histogram(actual, bins=bins)[0] / len(actual)
    
    # Avoid log(0)
    eps = 1e-8
    psi = np.sum((act_counts - exp_counts) * np.log((act_counts + eps) / (exp_counts + eps)))
    return psi

psi_vals = [compute_psi(train_age, serving) for _, serving in months]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(month_names, psi_vals,
               color=['#2dd4bf' if p < 0.1 else '#f97316' if p < 0.2 else '#ef4444' for p in psi_vals])
ax.axhline(0.1, color='#f97316', linestyle='--', alpha=0.7, label='Moderate shift (0.1)')
ax.axhline(0.2, color='#ef4444', linestyle='--', alpha=0.7, label='Significant shift (0.2)')
ax.set_ylabel('PSI')
ax.set_title('Population Stability Index over time', fontsize=12)
ax.legend(facecolor='#1a1d27', edgecolor='#2a2d3a')
plt.tight_layout()
plt.show()

### Validate: PSI rises past its alert thresholds under drift

The Population Stability Index bins the reference distribution and compares production
frequencies. Convention: PSI < 0.1 stable, 0.1–0.2 moderate, > 0.2 significant. We confirm
PSI stays low early and clears the 0.2 alarm on the major shift.

In [ ]:
for (name, _), p in zip(months, psi_vals):
    zone = 'stable' if p < 0.1 else ('moderate' if p < 0.2 else 'SIGNIFICANT')
    print(f'{name}: PSI={p:.3f} -> {zone}')
assert psi_vals[0] < 0.1, 'PSI stays low when the distribution barely moved'
assert psi_vals[-1] > 0.2, 'PSI clears the significant-shift threshold under major drift'
assert psi_vals[-1] > psi_vals[0], 'PSI grows monotonically with drift here'
print('\n✅ PSI is a single scalar drift alarm — the trigger to investigate or retrain')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **silent drift** | accuracy craters with no error thrown; add KS/PSI monitors (verified) |
| **PSI thresholds** | 0.1 moderate / 0.2 significant are conventions — tune to your risk |
| **accuracy ≠ calibration** | over-confident models mislead any probability-based decision (demo) |
| **train/serve skew** | the serving feature pipeline must match training exactly |
| **no retraining trigger** | wire drift alarms to a retraining/rollback decision |

Demo: an accurate model can still be badly miscalibrated.

In [ ]:
# Accuracy is NOT calibration. An over-confident model can be just as accurate as a
# well-calibrated one, yet its probabilities are systematically wrong — a disaster wherever
# you act on the probability (thresholds, expected value, risk). We measure calibration error
# (gap between predicted confidence and observed frequency) for both models.
# reliability_diagram returns (midpoints, accs, counts); accs[i] may be None for empty bins
def cal_error(probs, labels, n_bins=10):
    mid, accs, counts = reliability_diagram(probs, labels, n_bins)
    tot = sum(counts)
    return sum((counts[i] / tot) * abs(accs[i] - mid[i]) for i in range(len(mid)) if accs[i] is not None)
e_well = cal_error(pred_well, y_true)
e_over = cal_error(pred_over, y_true)
print(f'calibration error  well-calibrated model : {e_well:.3f}')
print(f'calibration error  over-confident model  : {e_over:.3f}')
assert e_over > e_well, 'the over-confident model is miscalibrated even if its accuracy is fine'
print('\nAn accurate model can still be badly calibrated -> monitor reliability, not just accuracy.')

## ✏️ Your turn

### Exercise 1: Implement calibration error (ECE)

Expected Calibration Error = weighted average of |accuracy − confidence| across bins.

In [ ]:
def expected_calibration_error(probs, labels, n_bins=10):
    """
    Compute Expected Calibration Error (ECE).
    
    ECE = Σ (n_bin/n_total) * |mean_confidence_bin - mean_accuracy_bin|
    
    Args:
        probs: np.ndarray (n,) predicted probabilities
        labels: np.ndarray (n,) true binary labels
        n_bins: int, number of probability bins
    Returns:
        float: ECE in [0, 1], lower is better calibrated
    """
    # TODO(you): bin predictions by confidence, compute weighted gap
    pass


ece_well = expected_calibration_error(pred_well, y_true)
ece_over = expected_calibration_error(pred_over, y_true)
print(f"Well-calibrated model ECE: {ece_well:.4f}")
print(f"Overconfident model ECE:   {ece_over:.4f}")

In [ ]:
ece_well = expected_calibration_error(pred_well, y_true)
ece_over = expected_calibration_error(pred_over, y_true)
assert ece_well is not None, "Should return a float"
assert 0 <= ece_well <= 1, "ECE should be between 0 and 1"
assert ece_well < ece_over, "Well-calibrated model should have lower ECE than overconfident"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def expected_calibration_error(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    n = len(probs)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() > 0:
            conf = probs[mask].mean()
            acc = labels[mask].mean()
            ece += (mask.sum() / n) * abs(conf - acc)
    return ece
```
</details>

### Exercise 2: Detect distribution drift using KS test

Implement a monitoring function that flags features with significant drift.

In [ ]:
def detect_drift(train_features, serving_features, alpha=0.05):
    """
    Detect which features have significant distribution drift.
    Uses the KS 2-sample test.
    
    Args:
        train_features: np.ndarray (n_train, n_features)
        serving_features: np.ndarray (n_serving, n_features)
        alpha: float, significance level for KS test
    Returns:
        list of int: feature indices where p-value < alpha (drift detected)
    """
    # TODO(you): for each feature, run scipy.stats.ks_2samp and check p-value
    pass


np.random.seed(42)
n_tr, n_se = 1000, 500
# Feature 0: no drift
f0_train = np.random.normal(0, 1, n_tr)
f0_serve = np.random.normal(0, 1, n_se)
# Feature 1: significant drift (mean shifted)
f1_train = np.random.normal(0, 1, n_tr)
f1_serve = np.random.normal(2, 1, n_se)

X_train_drift = np.column_stack([f0_train, f1_train])
X_serve_drift = np.column_stack([f0_serve, f1_serve])

drifted = detect_drift(X_train_drift, X_serve_drift)
print(f"Drifted features: {drifted}")

In [ ]:
drifted = detect_drift(X_train_drift, X_serve_drift)
assert drifted is not None, "Should return a list"
assert 1 in drifted, "Feature 1 (mean shifted by 2) should be flagged"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def detect_drift(train_features, serving_features, alpha=0.05):
    drifted = []
    for i in range(train_features.shape[1]):
        _, p_val = stats.ks_2samp(train_features[:, i], serving_features[:, i])
        if p_val < alpha:
            drifted.append(i)
    return drifted
```
</details>

## Key takeaways

- **Models decay silently:** drift moves the serving distribution away from training; detect
  it *before* accuracy craters.
- **Two drift alarms:** the **KS test** (distributional, with a p-value) and **PSI** (a
  single scalar with 0.1/0.2 thresholds) — both fire here, both stay quiet on stable data
  (verified).
- **Accuracy ≠ calibration:** an over-confident model can be accurate yet have wrong
  probabilities (demo) — monitor reliability wherever you act on the probability.
- **Instrument production** with drift + calibration monitors; they are your retraining
  triggers.